In [32]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys
import importlib

project_root = Path.cwd()
if not (project_root / 'metrics.py').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
import rsi_pipeline
importlib.reload(rsi_pipeline)
from rsi_pipeline import perform_rsi_analysis

In [28]:
intc = yf.download('INTC', start='2020-01-01')
ko = yf.download('KO', start='2020-01-01')


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [29]:
intc = perform_rsi_analysis(intc, label='INTC RSI')
ko = perform_rsi_analysis(ko, label='KO RSI')

Strategy Sharpe Ratio: 0.42
Buy and Hold Sharpe Ratio: 0.47
Strategy Sharpe Ratio: 1.12
Buy and Hold Sharpe Ratio: 0.70


In [35]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=('INTC', 'RSI'))

fig.add_trace(go.Scatter(x=intc.index, y=intc['Close'], name='INTC Close Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=intc.index, y=intc['RSI'], name='INTC RSI', line=dict(color='purple')), row=2, col=1)

fig.update_layout(title='INTC Price and RSI', xaxis_title='Date', yaxis_title='Price', yaxis2_title='RSI', template='plotly_dark', height=500,width=800)

fig.show()

In [40]:
fig = make_subplots(rows=1,cols=2, shared_yaxes=True,subplot_titles=('INTC', 'KO'))

fig.add_trace(go.Scatter(x=intc.index, y=intc['Equity_Curve'], name='INTC Equity Curve',mode='lines'), row=1, col=1)
fig.add_trace(go.Scatter(x=intc.index, y=intc['Buy_and_Hold'], name='INTC Buy and Hold',mode='lines'), row=1, col=1)

fig.add_trace(go.Scatter(x=ko.index, y=ko['Equity_Curve'], name='KO Equity Curve',mode='lines'), row=1, col=2)
fig.add_trace(go.Scatter(x=ko.index, y=ko['Buy_and_Hold'], name='KO Buy and Hold',mode='lines'), row=1, col=2)

fig.update_layout(title='Equity Curves', xaxis_title='Date', yaxis_title='Equity Curve', template='plotly_dark',width=1500)

fig.show()


# RSI Strategy Analysis — Conclusion

## What Was Tested
A standard RSI strategy was backtested on two stocks with 
different market characteristics:

- **INTC (2020–2026)** — choppy, volatile, declining then spiking
- **KO (2020–2026)** — slow, steady uptrend

**Strategy logic:**
- RSI < 30 → buy (oversold)
- RSI > 70 → exit (overbought)
- Window: 14 days (standard)

---

## Results

| | INTC Strategy | INTC Buy & Hold | KO Strategy | KO Buy & Hold |
|---|---|---|---|---|
| Sharpe | 0.42 | 0.47 | 1.12 | 0.7 |
| Outcome | Smoother ride | Volatile, spike | Missed uptrend | Steady growth |

---

## Conclusions

**1. RSI strategy protects capital in volatile markets**
On INTC, the strategy avoided the brutal 2022–2024 decline 
and the April 2026 spike entirely. The equity curve was 
significantly smoother than buy & hold, demonstrating strong 
downside protection on a choppy stock.

**2. RSI strategy misses gains in trending markets**
On KO, RSI rarely dropped below 30 — a stable uptrending stock 
seldom becomes genuinely oversold. The strategy sat in cash for 
long periods, missing the steady compounding that buy & hold captured.

**3. Strategy type must match market conditions**
This is the most important finding across the entire project:

| Market Type | Best Approach |
|-------------|--------------|
| Choppy / volatile | Active strategy (RSI, Mean Reversion, SMA) |
| Steady uptrend | Buy & Hold |
| Strong uptrend | Buy & Hold |

---

## Project-Wide Finding
Across all three strategies (SMA Crossover, Mean Reversion, RSI) 
tested on three stocks (AAPL, INTC, KO), one consistent pattern emerged:

**No active strategy beat buy & hold on trending stocks.
Every active strategy provided better downside protection on choppy stocks.**

This suggests that strategy selection should be driven by 
market regime detection — identifying whether a stock is 
trending or mean-reverting before choosing which strategy to apply.

---

## Key Learnings
- RSI works best as a mean-reversion signal on volatile stocks
- Stable stocks rarely hit RSI extremes — fewer signals, less edge
- Sharpe ratio alone is misleading — always read the equity curve
- Market regime matters more than strategy sophistication

---

## Next Steps
- Implement regime detection to switch strategies automatically
- Add maximum drawdown metric to all strategies
- Test RSI with trend filter now that base strategy is validated
- Research momentum strategies as a fourth strategy type